# Pipeline de Análise - Camada Gold
## Projeto Câmara Brasil

Este notebook gera métricas agregadas e visões analíticas a partir das tabelas Silver para servir dashboards e relatórios.

**Análises geradas:**
- Diversidade partidária por frente
- Engajamento de deputados (participação em frentes e votações)
- Padrões de votação
- Análise de despesas por tipo, deputado e região
- Ranking de fornecedores
- Frequência de eventos
- Detecção de anomalias em despesas

**Entrada:** workspace.default.silver_camara_*

**Saída:** workspace.default.gold_camara_*

In [0]:
from pyspark.sql import DataFrame, SparkSession
from pyspark.sql import functions as F
from pyspark.sql.window import Window
from pyspark.sql.types import DoubleType
from typing import Optional, List

# Configurações Unity Catalog
CATALOG = "workspace"
SCHEMA = "default"
SILVER_PREFIX = "silver_camara_"
GOLD_PREFIX = "gold_camara_"

print("✓ Configuração carregada")
print(f"  Catálogo UC: {CATALOG}.{SCHEMA}")
print(f"  Origem: {SILVER_PREFIX}*")
print(f"  Destino: {GOLD_PREFIX}*")

In [0]:
def read_silver(table_name: str) -> DataFrame:
    """Lê tabela da camada Silver."""
    full_table = f"{CATALOG}.{SCHEMA}.{SILVER_PREFIX}{table_name}"
    return spark.read.table(full_table)

def save_gold(df: DataFrame, table_name: str, partition_by: Optional[List[str]] = None) -> None:
    """Salva DataFrame na camada Gold com Delta Lake."""
    full_table = f"{CATALOG}.{SCHEMA}.{GOLD_PREFIX}{table_name}"
    
    writer = df.write.format("delta").mode("overwrite")
    
    if partition_by:
        writer = writer.partitionBy(*partition_by)
    
    writer.saveAsTable(full_table)
    count = df.count()
    print(f"✓ Tabela '{full_table}' gravada: {count:,} registros")

print("✓ Funções auxiliares carregadas")

In [0]:
def diversidade_partidaria_frentes() -> DataFrame:
    """Calcula diversidade partidária por frente usando índice de Simpson."""
    print("Calculando diversidade partidária das frentes...")
    
    # Ler dados
    membros_df = read_silver("fact_frentes_membros")
    deputados_df = read_silver("dim_deputados")
    frentes_df = read_silver("dim_frentes")
    
    # Join para obter partido de cada membro
    membros_com_partido = membros_df.join(
        deputados_df.select("id", "sigla_partido"),
        membros_df["id"] == deputados_df["id"],
        "left"
    )
    
    # Contar membros por frente e partido
    contagem = membros_com_partido.groupBy("idFrente", "sigla_partido").agg(
        F.count("*").alias("qtde_membros")
    )
    
    # Calcular total de membros por frente
    total_por_frente = contagem.groupBy("idFrente").agg(
        F.sum("qtde_membros").alias("total_membros")
    )
    
    # Calcular índice de Simpson (diversidade)
    # Simpson = 1 - sum((n/N)^2) onde n=membros do partido, N=total
    diversidade = contagem.join(total_por_frente, "idFrente")
    diversidade = diversidade.withColumn(
        "proporcao", F.col("qtde_membros") / F.col("total_membros")
    )
    diversidade = diversidade.withColumn(
        "simpson_parcial", F.col("proporcao") * F.col("proporcao")
    )
    
    indice_diversidade = diversidade.groupBy("idFrente").agg(
        (F.lit(1) - F.sum("simpson_parcial")).alias("indice_diversidade"),
        F.first("total_membros").alias("total_membros"),
        F.count("sigla_partido").alias("qtde_partidos")
    )
    
    # Join com nome da frente
    resultado = indice_diversidade.join(
        frentes_df.select("id", "titulo"),
        indice_diversidade["idFrente"] == frentes_df["id"],
        "left"
    ).select(
        "idFrente",
        "titulo",
        "indice_diversidade",
        "qtde_partidos",
        "total_membros"
    ).orderBy(F.desc("indice_diversidade"))
    
    save_gold(resultado, "diversidade_partidaria_frentes")
    print(f"  Frentes analisadas: {resultado.count()}")
    return resultado

print("✓ Função diversidade_partidaria_frentes carregada")

In [0]:
def engajamento_deputados() -> DataFrame:
    """Calcula score de engajamento dos deputados."""
    print("Calculando engajamento dos deputados...")
    
    # Ler dados
    deputados_df = read_silver("dim_deputados")
    membros_df = read_silver("fact_frentes_membros")
    votos_df = read_silver("fact_votacoes")
    
    # Contar frentes por deputado
    frentes_por_dep = membros_df.groupBy("id").agg(
        F.countDistinct("idFrente").alias("qtde_frentes")
    )
    
    # Contar votações por deputado
    votos_por_dep = votos_df.groupBy("deputado_id").agg(
        F.count("idVotacao").alias("qtde_votacoes")
    )
    
    # Join tudo
    engajamento = deputados_df.select("id", "nome", "sigla_partido", "sigla_uf")
    engajamento = engajamento.join(frentes_por_dep, "id", "left")
    engajamento = engajamento.join(
        votos_por_dep,
        engajamento["id"] == votos_por_dep["deputado_id"],
        "left"
    )
    
    # Preencher nulls com 0
    engajamento = engajamento.fillna(0, ["qtde_frentes", "qtde_votacoes"])
    
    # Calcular score normalizado (0-100)
    # Score = média ponderada de frentes (40%) e votações (60%)
    max_frentes = engajamento.agg(F.max("qtde_frentes")).collect()[0][0]
    max_votos = engajamento.agg(F.max("qtde_votacoes")).collect()[0][0]
    
    if max_frentes and max_frentes > 0:
        engajamento = engajamento.withColumn(
            "score_frentes", (F.col("qtde_frentes") / F.lit(max_frentes)) * 40
        )
    else:
        engajamento = engajamento.withColumn("score_frentes", F.lit(0))
    
    if max_votos and max_votos > 0:
        engajamento = engajamento.withColumn(
            "score_votacoes", (F.col("qtde_votacoes") / F.lit(max_votos)) * 60
        )
    else:
        engajamento = engajamento.withColumn("score_votacoes", F.lit(0))
    
    engajamento = engajamento.withColumn(
        "score_engajamento",
        (F.col("score_frentes") + F.col("score_votacoes")).cast(DoubleType())
    )
    
    resultado = engajamento.select(
        "id",
        "nome",
        "sigla_partido",
        "sigla_uf",
        "qtde_frentes",
        "qtde_votacoes",
        "score_engajamento"
    ).orderBy(F.desc("score_engajamento"))
    
    save_gold(resultado, "engajamento_deputados")
    print(f"  Deputados analisados: {resultado.count()}")
    return resultado

print("✓ Função engajamento_deputados carregada")

In [0]:
def analise_despesas_tipo() -> DataFrame:
    """Analisa despesas agregadas por tipo."""
    print("Analisando despesas por tipo...")
    
    despesas_df = read_silver("fact_despesas")
    
    analise = despesas_df.groupBy("tipoDespesa").agg(
        F.sum("valor_liquido").alias("total_gasto"),
        F.avg("valor_liquido").alias("media_gasto"),
        F.count("*").alias("qtde_transacoes"),
        F.countDistinct("idDeputado").alias("qtde_deputados")
    ).orderBy(F.desc("total_gasto"))
    
    save_gold(analise, "despesas_por_tipo")
    print(f"  Tipos de despesa analisados: {analise.count()}")
    return analise

print("✓ Função analise_despesas_tipo carregada")

In [0]:
def analise_despesas_deputado() -> DataFrame:
    """Analisa despesas por deputado."""
    print("Analisando despesas por deputado...")
    
    despesas_df = read_silver("fact_despesas")
    deputados_df = read_silver("dim_deputados")
    
    despesas_dep = despesas_df.groupBy("idDeputado").agg(
        F.sum("valor_liquido").alias("total_gasto"),
        F.avg("valor_liquido").alias("media_gasto"),
        F.count("*").alias("qtde_despesas")
    )
    
    resultado = despesas_dep.join(
        deputados_df.select("id", "nome", "sigla_partido", "sigla_uf"),
        despesas_dep["idDeputado"] == deputados_df["id"],
        "left"
    ).select(
        F.col("idDeputado").alias("deputado_id"),
        "nome",
        "sigla_partido",
        "sigla_uf",
        "total_gasto",
        "media_gasto",
        "qtde_despesas"
    ).orderBy(F.desc("total_gasto"))
    
    save_gold(resultado, "despesas_por_deputado")
    print(f"  Deputados analisados: {resultado.count()}")
    return resultado

print("✓ Função analise_despesas_deputado carregada")

In [0]:
def analise_despesas_regiao() -> DataFrame:
    """Analisa despesas por UF."""
    print("Analisando despesas por região...")
    
    despesas_df = read_silver("fact_despesas")
    deputados_df = read_silver("dim_deputados")
    
    # Join para obter UF
    despesas_com_uf = despesas_df.join(
        deputados_df.select("id", "sigla_uf"),
        despesas_df["idDeputado"] == deputados_df["id"],
        "left"
    )
    
    analise = despesas_com_uf.groupBy("sigla_uf").agg(
        F.sum("valor_liquido").alias("total_gasto"),
        F.avg("valor_liquido").alias("media_gasto"),
        F.count("*").alias("qtde_despesas"),
        F.countDistinct("idDeputado").alias("qtde_deputados")
    ).orderBy(F.desc("total_gasto"))
    
    save_gold(analise, "despesas_por_uf")
    print(f"  UFs analisadas: {analise.count()}")
    return analise

print("✓ Função analise_despesas_regiao carregada")

In [0]:
def ranking_fornecedores(top_n: int = 50) -> DataFrame:
    """Gera ranking dos maiores fornecedores."""
    print(f"Gerando ranking top {top_n} fornecedores...")
    
    despesas_df = read_silver("fact_despesas")
    
    ranking = despesas_df.filter(F.col("nomeFornecedor").isNotNull()).groupBy("nomeFornecedor").agg(
        F.sum("valor_liquido").alias("total_recebido"),
        F.count("*").alias("qtde_transacoes"),
        F.countDistinct("idDeputado").alias("qtde_deputados_atendidos")
    ).orderBy(F.desc("total_recebido")).limit(top_n)
    
    save_gold(ranking, "ranking_fornecedores")
    print(f"  Fornecedores no ranking: {ranking.count()}")
    return ranking

print("✓ Função ranking_fornecedores carregada")

In [0]:
def analise_votacoes() -> DataFrame:
    """Analisa padrões de votação."""
    print("Analisando padrões de votação...")
    
    votos_df = read_silver("fact_votacoes")
    
    # Análise por tipo de voto
    analise = votos_df.groupBy("tipoVoto").agg(
        F.count("*").alias("qtde_votos"),
        F.countDistinct("deputado_id").alias("qtde_deputados"),
        F.countDistinct("idVotacao").alias("qtde_votacoes")
    ).orderBy(F.desc("qtde_votos"))
    
    save_gold(analise, "padroes_votacao")
    print(f"  Tipos de voto analisados: {analise.count()}")
    return analise

print("✓ Função analise_votacoes carregada")

In [0]:
def analise_eventos_temporal() -> DataFrame:
    """Analisa frequência de eventos por mês."""
    print("Analisando frequência de eventos...")
    
    eventos_df = read_silver("fact_eventos")
    
    # Filtrar eventos com data válida
    eventos_com_data = eventos_df.filter(F.col("data_inicio").isNotNull())
    
    # Agrupar por ano e mês
    frequencia = eventos_com_data.groupBy("ano", "mes").agg(
        F.count("*").alias("qtde_eventos"),
        F.countDistinct("descricaoTipo").alias("qtde_tipos_eventos")
    ).orderBy("ano", "mes")
    
    save_gold(frequencia, "eventos_por_mes")
    print(f"  Períodos analisados: {frequencia.count()}")
    return frequencia

print("✓ Função analise_eventos_temporal carregada")

In [0]:
def detectar_anomalias_despesas() -> DataFrame:
    """Detecta anomalias em despesas usando z-score."""
    print("Detectando anomalias em despesas...")
    
    despesas_df = read_silver("fact_despesas")
    deputados_df = read_silver("dim_deputados")
    
    # Join para obter dados do deputado
    despesas_com_deputado = despesas_df.join(
        deputados_df.select("id", "nome", "sigla_partido", "sigla_uf"),
        despesas_df["idDeputado"] == deputados_df["id"],
        "left"
    )
    
    # Calcular média e desvio padrão por tipo de despesa
    stats = despesas_df.groupBy("tipoDespesa").agg(
        F.avg("valor_liquido").alias("media"),
        F.stddev("valor_liquido").alias("desvio_padrao")
    )
    
    # Join e calcular z-score
    com_stats = despesas_com_deputado.join(stats, "tipoDespesa", "left")
    com_stats = com_stats.withColumn(
        "z_score",
        F.when(
            (F.col("desvio_padrao").isNotNull()) & (F.col("desvio_padrao") > 0),
            (F.col("valor_liquido") - F.col("media")) / F.col("desvio_padrao")
        ).otherwise(0)
    )
    
    # Filtrar anomalias (|z-score| > 3)
    anomalias = com_stats.filter(
        (F.abs(F.col("z_score")) > 3) & (F.col("valor_liquido") > 0)
    ).select(
        F.col("idDeputado").alias("deputado_id"),
        "nome",
        "sigla_partido",
        "sigla_uf",
        "tipoDespesa",
        "valor_liquido",
        "media",
        "z_score",
        "data_documento"
    ).orderBy(F.desc(F.abs(F.col("z_score"))))
    
    save_gold(anomalias, "anomalias_despesas")
    print(f"  Anomalias detectadas: {anomalias.count()}")
    return anomalias

print("✓ Função detectar_anomalias_despesas carregada")

In [0]:
def analyze_all() -> None:
    """Executa todas as análises da camada Gold."""
    print("="*60)
    print("INICIANDO PIPELINE DE ANÁLISE - CAMADA GOLD")
    print("="*60)
    
    try:
        # Análises
        print("\n[1/9] Diversidade partidária...")
        diversidade_partidaria_frentes()
        
        print("\n[2/9] Engajamento de deputados...")
        engajamento_deputados()
        
        print("\n[3/9] Despesas por tipo...")
        analise_despesas_tipo()
        
        print("\n[4/9] Despesas por deputado...")
        analise_despesas_deputado()
        
        print("\n[5/9] Despesas por região...")
        analise_despesas_regiao()
        
        print("\n[6/9] Ranking de fornecedores...")
        ranking_fornecedores()
        
        print("\n[7/9] Padrões de votação...")
        analise_votacoes()
        
        print("\n[8/9] Frequência de eventos...")
        analise_eventos_temporal()
        
        print("\n[9/9] Detecção de anomalias...")
        detectar_anomalias_despesas()
        
        print("\n" + "="*60)
        print("✓ PIPELINE GOLD CONCLUÍDO COM SUCESSO")
        print("="*60)
        
        # Estatísticas finais
        print("\n📊 Resumo das tabelas Gold criadas:")
        tables = [
            "diversidade_partidaria_frentes",
            "engajamento_deputados",
            "despesas_por_tipo",
            "despesas_por_deputado",
            "despesas_por_uf",
            "ranking_fornecedores",
            "padroes_votacao",
            "eventos_por_mes",
            "anomalias_despesas"
        ]
        
        for table in tables:
            full_table = f"{CATALOG}.{SCHEMA}.{GOLD_PREFIX}{table}"
            try:
                count = spark.read.table(full_table).count()
                print(f"  ✓ {table}: {count:,} registros")
            except Exception as e:
                print(f"  ✗ {table}: erro ao contar")
        
    except Exception as e:
        print(f"\n❌ ERRO NO PIPELINE: {str(e)}")
        import traceback
        traceback.print_exc()
        raise

# Executar pipeline
analyze_all()